In [1]:
!pip install --upgrade pyproject-hooks build


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install --use-pep517 kaggle

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for kaggle: filename=kaggle-1.6.17-py3-none-any.whl size=105838 sha256=1f46b3fe2024941d10ad65b234853f7ac2bbbaefafeebf33406ef649bc853406
  Stored in directory: /tmp/pip-ephem-wheel-cache-zhvz6_nm/wheels/2b/af/a9/70bffa2773af622d2ebea9c8d407720b86e67bd40c465bf837
Successfully built kaggle

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import os

# Crea la carpeta .kaggle en el home si no existe
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

# Mueve el kaggle.json subido a ~/.kaggle/kaggle.json
!mv kaggle.json ~/.kaggle/kaggle.json

# Ajusta permisos a 600
!chmod 600 ~/.kaggle/kaggle.json

print("kaggle.json configurado en ~/.kaggle/")


mv: cannot stat 'kaggle.json': No such file or directory
kaggle.json configurado en ~/.kaggle/


In [4]:
!rm -rf /tmp/data/*
!echo "✅ Archivos antiguos eliminados."


✅ Archivos antiguos eliminados.


In [5]:
# 1️⃣ Crear la carpeta de trabajo
!mkdir -p /tmp/data

# 2️⃣ Descargar el dataset completo
!kaggle datasets download -d marcosdiezz/consumo-electrico-0 -p /tmp/data

# 3️⃣ Descomprimir solo el archivo deseado
!unzip -o "/tmp/data/consumo-electrico-0.zip" "divididof/consumo_parte_116.csv" -d /tmp/data

# 4️⃣ Mover el archivo al directorio raíz
!mv /tmp/data/divididof/consumo_parte_116.csv /tmp/data/

# 5️⃣ Eliminar los archivos no deseados para limpiar espacio
!rm -rf /tmp/data/divididof /tmp/data/consumo-electrico-0.zip

# 6️⃣ Verificar que solo queda el archivo correcto
!ls -lh /tmp/data

Dataset URL: https://www.kaggle.com/datasets/marcosdiezz/consumo-electrico-0
License(s): unknown
100%|█████████████████████████████████████▉| 6.20G/6.20G [06:43<00:00, 19.0MB/s]
100%|██████████████████████████████████████| 6.20G/6.20G [06:43<00:00, 16.5MB/s]
Archive:  /tmp/data/consumo-electrico-0.zip
  inflating: /tmp/data/divididof/consumo_parte_116.csv  
total 35M
-rw-r--r-- 1 jovyan users 35M Feb 26 13:45 consumo_parte_116.csv


In [11]:
import mlrun
import pandas as pd

# Ruta del dataset seleccionado
csv_path = "/tmp/data/consumo_parte_116.csv"

# Cargar el CSV en un DataFrame
df = pd.read_csv(csv_path)

# Crear un contexto de MLRun
context = mlrun.get_or_create_ctx("kaggle-ingest")

# Loggear el dataset en MLRun para que se guarde en MinIO
context.log_dataset("consumo_electrico", df=df, format="csv")

print("✅ Dataset `consumo_parte_116.csv` subido a MLRun y MinIO.")


✅ Dataset `consumo_parte_116.csv` subido a MLRun y MinIO.


In [12]:

%%writefile preprocess.py
import mlrun
import pandas as pd

def preprocess_data(context, file_path: str):
    """Preprocesa el dataset de consumo eléctrico."""
    
    # Cargar el dataset
    df = pd.read_csv(file_path)

    # Convertir la columna fecha a formato datetime
    df["fecha"] = pd.to_datetime(df["fecha"])

    # Eliminar filas con valores nulos
    df.dropna(inplace=True)

    # Filtrar valores negativos en consumo_kwh y coste_euros
    df = df[(df["consumo_kwh"] > 0) & (df["coste_euros"] > 0)]

    # Loggear el dataset preprocesado en MLRun
    context.log_dataset("consumo_electrico_preprocesado", df=df, format="csv")

    context.logger.info(f"✅ Preprocesamiento completado. Filas finales: {len(df)}")


Overwriting preprocess.py


In [13]:
import mlrun
from mlrun import code_to_function

# Crear el proyecto en MLRun
project = mlrun.get_or_create_project("consumo-elec-project", context="./")

# Registrar la función de preprocesamiento
preprocess_func = code_to_function(
    name="preprocess-data",
    filename="preprocess.py",
    handler="preprocess_data",
    kind="job",
    image="mlrun/mlrun"
)

project.set_function(preprocess_func)
project.save()

> 2025-02-26 18:42:36,921 [info] Project loaded successfully: {"project_name":"consumo-elec-project"}


In [14]:
run = preprocess_func.run(
    inputs={"file_path": "/tmp/data/consumo_parte_116.csv"},
    local=True
)

> 2025-02-26 18:42:38,692 [warning] it is recommended to use k8s secret (specify secret_name), specifying the aws_access_key/aws_secret_key directly is unsafe
> 2025-02-26 18:42:38,723 [info] Storing function: {"db":"http://mlrun-api:8080","name":"preprocess-data-preprocess-data","uid":"3d996eae4fee4b2184851d13b6773f6c"}
> 2025-02-26 18:42:42,269 [info] ✅ Preprocesamiento completado. Filas finales: 943357


project,uid,iter,start,state,kind,name,labels,inputs,parameters,results,artifacts
consumo-elec-project,...b6773f6c,0,Feb 26 18:42:38,completed,run,preprocess-data-preprocess-data,kind=localowner=jovyanhost=mlrun-jupyter-7d5ccd7965-5bgtf,file_path,,,consumo_electrico_preprocesado


> 2025-02-26 18:42:42,352 [info] Run execution finished: {"name":"preprocess-data-preprocess-data","status":"completed"}
